In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [2]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [3]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [4]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [5]:
from pyspark.sql.functions import col, window, avg, asc, round as _round

df_gdansk = df.filter(col("store") == "Gdańsk")

gdansk_najg_godz = (
    df_gdansk.groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("srednia_kwota"))
    .orderBy(asc("srednia_kwota")) # Sortujemy rosnąco
)

(
    gdansk_najg_godz.select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia_kwota"
    )
    .show(1, truncate=False)
)

+-------------------+-------------------+-------------+
|od                 |do                 |srednia_kwota|
+-------------------+-------------------+-------------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01       |
+-------------------+-------------------+-------------+
only showing top 1 row



In [6]:
from pyspark.sql.functions import count, date_format

kategorie_w_oknie = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(count("tx_id").alias("liczba_tx"))
)

wynik_09_00 = (
    kategorie_w_oknie.filter(date_format(col("window.start"), "HH:mm:ss") == "09:00:00")
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "category",
        "liczba_tx"
    )
    .orderBy("category")
)

wynik_09_00.show(truncate=False)

+-------------------+-------------------+-----------+---------+
|od                 |do                 |category   |liczba_tx|
+-------------------+-------------------+-----------+---------+
|2026-04-12 09:00:00|2026-04-12 09:30:00|elektronika|611      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|książki    |622      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|odzież     |605      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|żywność    |567      |
+-------------------+-------------------+-----------+---------+



In [7]:
from pyspark.sql.functions import desc

okna_15m = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .orderBy(desc("liczba_tx"))
)

(
    okna_15m.select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx"
    )
    .show(1, truncate=False)
)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
+-------------------+-------------------+---------+
only showing top 1 row

